# Predicting Diagnostic Delay Signals from Patient-Authored Text (Reddit/Health Forums)
### Modeling the 'Diagnostic Odyssey' Using Longitudinal NLP and Post-Hoc Explainability

---

## 1. Project Overview & Research Motivation
Traditional diagnostic delay studies rely heavily on structured **Electronic Health Records (EHR)** or claims databases. However, EHR data suffers from an inherent **left-censoring bias**: medical records only begin *after* a patient enters a formal clinical setting.

In contrast, patients experiencing the **'Diagnostic Odyssey'** (especially across complex autoimmune conditions like Lupus, Sjögren's, and UCTD) frequently document their evolving symptoms, physician dismissals, anxiety, and delays publicly on online health communities (**Reddit: `r/lupus`, `r/Sjogrens`, `r/AutoimmuneDisease`, `r/ChronicIllness`**) months or years before receiving a clinical diagnosis.

### Core Research Questions
1. Can **longitudinal language patterns** and posting dynamics predict whether a patient is on a trajectory toward severe diagnostic delay (>1 year)?
2. Which **early linguistic signals** (medical dismissal/gaslighting, multisystem symptom breadth, sentiment drift) are most predictive?
3. Can we provide **faithful post-hoc clinical explanations** (post-level attention weights, SHAP feature attributions, LIME token highlights) for model predictions?

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.data_collection.synthetic_generator import SyntheticPatientGenerator
from src.preprocessing.anonymizer import PatientAnonymizer
from src.preprocessing.labeler import DiagnosticDelayLabeler
from src.preprocessing.timeline_builder import PatientTimelineBuilder
from src.feature_engineering.lexicon_extractor import DomainLexiconExtractor
from src.feature_engineering.temporal_features import TemporalFeatureExtractor
from src.feature_engineering.sentiment_trajectory import SentimentTrajectoryExtractor
from src.feature_engineering.text_encoder import LongitudinalTextEncoder
from src.models.baselines import BaselineModelSuite
from src.models.bilstm_attention import LongitudinalBiLSTMAttention, BiLSTMTrainer, PatientTimelineDataset
from src.explainability.shap_explainer import DiagnosticDelaySHAPExplainer
from src.explainability.attention_visualizer import TimelineAttentionVisualizer
from src.explainability.case_study_generator import DiagnosticCaseStudyGenerator

print("Environment and modules initialized successfully!")

## 2. Data Ingestion & AoIR-Compliant Anonymization
In accordance with the **Association of Internet Researchers (AoIR)** ethical guidelines:
- Direct author handles are securely hashed with a salted SHA-256 function (`PATIENT_...`).
- All clinical entities, clinic names, URLs, phone numbers, and emails are scrubbed.
- Quotes are aggregated or paraphrased to prevent direct re-identification.

In [ ]:
# Generate realistic longitudinal patient cohort
generator = SyntheticPatientGenerator(seed=42)
raw_df = generator.generate_dataset(num_patients=100, long_delay_ratio=0.55)

# Apply salted SHA-256 pseudonymization and PII scrubbing
anonymizer = PatientAnonymizer(salt="ResearchStudySalt2026")
anon_df = anonymizer.anonymize_dataframe(raw_df, author_col="author")

print(f"Total anonymized submissions: {len(anon_df)}")
print(f"Unique patients: {anon_df['patient_id'].nunique()}")
anon_df[['patient_id', 'subreddit', 'text']].head(3)

## 3. Weak Supervision Timeline Extraction & Labeling
We extract self-reported diagnostic durations via validated regex heuristic patterns (e.g., *'took me 3 years to get diagnosed'*, *'finally diagnosed after 18 months'*).
- **Class 0 (Short Delay)**: $\le 1.0\text{ year}$
- **Class 1 (Long Delay)**: $> 1.0\text{ year}$

In [ ]:
labeler = DiagnosticDelayLabeler(delay_threshold_years=1.0)
labeled_summary = labeler.label_dataset(anon_df, patient_col="patient_id", text_col="text", created_utc_col="created_utc")

# Merge labels back
merged_df = anon_df.merge(
    labeled_summary[["patient_id", "label", "delay_duration_years", "evidence_snippet"]],
    on="patient_id",
    how="left"
)

print("Label distribution:")
print(labeled_summary['label'].value_counts(dropna=False))

## 4. Longitudinal Timeline Construction (No Data Leakage)
The final post containing the explicit diagnostic milestone announcement is **pruned** from model input sequences to strictly guarantee true pre-diagnostic predictive validity.

In [ ]:
timeline_builder = PatientTimelineBuilder(min_posts=2, max_posts=15, exclude_labeling_post=True)
timelines = timeline_builder.build_patient_timelines(
    merged_df,
    patient_col="patient_id",
    text_col="text",
    timestamp_col="created_utc",
    label_col="label"
)

valid_timelines = [t for t in timelines if t['label'] is not None]
print(f"Constructed {len(valid_timelines)} pure pre-diagnostic sequence timelines.")

## 5. Multi-Modal Feature Extraction
We extract:
1. **Medical Dismissal & Gaslighting Signals** (*'in my head'*, *'just stress'*, *'refused to test'*)
2. **Multi-Organ Symptom Clusters** (Musculoskeletal, Systemic, Mucocutaneous, Neurological, GI)
3. **Temporal Dynamics** (Inter-post intervals, posting velocity, burstiness index)
4. **Sentiment & Emotional Trajectories** (Frustration slope, polarity drift)
5. **Dense Biomedical Embeddings** (`Sentence-BERT` / `ClinicalBERT`)

In [ ]:
lex_ext = DomainLexiconExtractor()
temp_ext = TemporalFeatureExtractor()
sent_ext = SentimentTrajectoryExtractor()
text_enc = LongitudinalTextEncoder()

tabular_rows = []
y_list = []
seq_tensors = []
seq_masks = []

for t in valid_timelines:
    texts = t["texts"]
    ts = t["timestamps"]
    deltas = t["delta_days"]
    
    # Features
    l_feat = lex_ext.extract_features_from_timeline(texts)
    tm_feat = temp_ext.extract_features(ts, deltas)
    s_feat = sent_ext.extract_trajectory_features(texts)
    
    combined = {**l_feat, **tm_feat, **s_feat}
    tabular_rows.append(combined)
    y_list.append(t["label"])
    
    # Post embeddings + extras
    embs = text_enc.encode_texts(texts)
    extras = np.array([
        [lex_ext.extract_features_from_text(p)["dismissal_count"], deltas[i]]
        for i, p in enumerate(texts)
    ], dtype=np.float32)
    
    full_p = np.concatenate([embs, extras], axis=-1)
    padded = np.zeros((15, full_p.shape[1]), dtype=np.float32)
    mask = np.zeros(15, dtype=bool)
    act_len = min(15, len(full_p))
    padded[-act_len:] = full_p[-act_len:]
    mask[-act_len:] = True
    
    seq_tensors.append(padded)
    seq_masks.append(mask)

X_tabular = pd.DataFrame(tabular_rows).values
feature_names = list(pd.DataFrame(tabular_rows).columns)
y_arr = np.array(y_list, dtype=int)

print(f"Extracted {X_tabular.shape[1]} tabular engineered features across {len(y_arr)} patients.")

## 6. Baseline Classifiers & Longitudinal BiLSTM-Attention
We benchmark classical baselines (Logistic Regression, SVM, Random Forest) against our **Longitudinal BiLSTM with Bahdanau Timeline Attention**.

In [ ]:
# Classical Baselines
baselines = BaselineModelSuite(random_state=42)
cv_results = baselines.fit_and_evaluate_cv(X_tabular, y_arr, n_splits=5)
print("=== Baseline Performance (5-Fold CV) ===")
print(cv_results.to_string(index=False))

# Longitudinal BiLSTM + Attention
dataset = PatientTimelineDataset(seq_tensors, y_list, seq_masks)
loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)
eval_loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=False)

trainer = BiLSTMTrainer(input_dim=seq_tensors[0].shape[1], hidden_dim=64)
for ep in range(12):
    loss = trainer.train_epoch(loader)

probs, preds, _ = trainer.evaluate(eval_loader)
print(f"\nBiLSTM Attention Training Complete (Final Loss: {loss:.4f})")

## 7. Explainability: SHAP Feature Importance & Timeline Attention
We compute SHAP attributions on domain features and extract post-level attention weights $\alpha_t$ to pinpoint turning points in individual patient odysseys.

In [ ]:
# Global SHAP Feature Importance
rf = baselines.fitted_models["Random_Forest"]
shap_exp = DiagnosticDelaySHAPExplainer(model=rf, feature_names=feature_names)
shap_exp.plot_summary(X_tabular)

# Timeline Attention for Sample Patient
sample_idx = 0
p_t = valid_timelines[sample_idx]
p_attn = trainer.get_attention_for_patient(seq_tensors[sample_idx], seq_masks[sample_idx])

TimelineAttentionVisualizer.plot_patient_attention(
    patient_id=p_t["patient_id"],
    posts_text=p_t["texts"],
    cumulative_days=p_t["cumulative_days"],
    attention_weights=p_attn,
    predicted_prob=float(probs[sample_idx]),
    true_label=p_t["label"]
)